In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl
import matplotlib.dates as mdates
import matplotlib
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import seaborn as sns
#import plotly.io as pio
#pio.renderers.default = 'browser'
#pio.renderers.default = 'notebook'

In [ ]:
%run ../age_model/20260215_read_data.py

In [ ]:

pairs = []

for path, g in df.groupby("path_name", sort=False):

    ends = g.loc[g["age_end"].notna(),
                 ["index", "sample", "path_num", "age_end", "age2", "age_absolute", "d13c", "d18o"]]

    starts = g.loc[g["age2"].notna(),
                   ["index", "sample", "path_num", "age_end", "age2", "age_absolute", "d13c", "d18o"]]

    m = ends.merge(
        starts,
        left_on="age_end",
        right_on="age2",
        how="inner",
        suffixes=("_end", "_start")
    )

    if not m.empty:
        m.insert(0, "path_name", path)
        pairs.append(m)

dark_bands = pd.concat(pairs, ignore_index=True)
dark_bands = dark_bands[dark_bands['index_end'] > dark_bands['index_start']].reset_index(drop=True)
dark_bands

In [ ]:
%run ../../2025-08-22_FredsData-post2008/fred_data.py

In [ ]:
tide = pd.read_csv('../../2025-10-10_Fred-TideGauge/memo_edited.csv')
tide['age_dt'] = decimal_year_to_datetime(tide['Date'])#.dt.date
tide = tide.set_index('age_dt')
tide

In [ ]:
enso_phase = pd.read_csv('../../ENSO_Phase_Classification.csv')
enso_phase.columns=['Year','Phase']
enso_phase['Date'] = decimal_year_to_datetime(enso_phase['Year'])
#enso_phase.set_index('Date', inplace=True)
enso_phase['Phase'] = enso_phase['Phase'].str.replace('Warm Phase', 'W').str.replace('Cold Phase', 'C').str.replace('Neutral Phase','N')


enso = pd.read_csv("../../ENSO_Time_Series.csv", parse_dates=[0])
#enso = enso.set_index("Date")
enso['day_of_year'] = enso['Date'].dt.dayofyear
enso['is_leap'] = enso['Date'].dt.is_leap_year
enso['total_days'] = np.where(enso['is_leap'], 366, 365)
enso['decimal_year'] = enso['Date'].dt.year + (enso['day_of_year'] - 1) / enso['total_days']


margin = 0.5
enso['phase'] = 'N'
enso.loc[enso['ENSO']>=margin,'phase'] = "C"
enso.loc[enso['ENSO']<=-margin,'phase'] = "W"
enso['phase_group'] = (enso['phase'] != enso['phase'].shift()).cumsum()
blocks = enso.groupby('phase_group').agg(
    start=('Date', 'first'),
    end=('Date', 'last'),
    value=('phase', 'first')
).reset_index(drop=True)

ENSO_match = np.array([[1982.2, 1984.1],
                       [1986.75, 1988.3],
                       [1992.1, 1992.96],
                       [1997.2, 1998.8]])

enso_fft_freq, enso_fft = FFT_res(enso['ENSO'], 30)


enso


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c'],  mode="markers", marker_color="lightgreen",  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)


#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')



for i, index in enumerate(blocks.index):
    block = blocks.loc[index]
    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
                  x1=datetime_to_decimal_year(block['end']), 
                  fillcolor=phase_color[block['value']], opacity=0.1)

fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.975], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
                        name='M8.1 EQ 2007 Apr'))

fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="Standard Deviation", secondary_y=True)

fig.show()
#fig.write_html('d13c_std.html')


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
#fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c'],  mode="markers", marker_color="lightgreen",  text=df_sorted['Name'],  name='d13C'),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c_no_tectonic'],  mode="markers", marker_color="lightgreen",  name='d13C without tectonic', text=df_sorted['Name']),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_notec_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)



#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)

#fig.add_trace(go.Scatter(x=coefs_centered['age_absolute'], y=coefs_centered['d13c_slope'], mode="lines", line_color='pink', name='derivative'), secondary_y=True)
fig.add_trace(go.Scatter(x=tide['Date'], y=tide['Sea Level'], mode="lines", line_color='#F54927', name='Tide Gauge', line_width=1), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=False)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=True)) # Grid for secondary Y-axis)


#for i, dates in enumerate(ENSO_match):
#    fig.add_vrect(x0=dates[0], 
#                  x1=dates[1], 
#                  fillcolor=phase_color['C'], opacity=0.1)

for i, dates in enumerate(ENSO_match):
    fig.add_vrect(x0=dates[0], 
                  x1=dates[1], 
                  fillcolor=phase_color['C'], opacity=0.1)
    
#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

#for i, index in enumerate(blocks.index):
#    block = blocks.loc[index]
#    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
#                  x1=datetime_to_decimal_year(block['end']), 
#                  fillcolor=phase_color[block['value']], opacity=0.1)

#fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.975], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
#                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
#                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
#                        name='M8.1 EQ 2007 Apr'))

#fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

#fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d13C and tide gauge", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="Tide Gauge [mm]", secondary_y=True)

fig.show()
#fig.write_html('d13c_notecto.html')


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c'],  mode="markers", marker_color="lightgreen",  text=df_sorted['Name'],  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma2'], mode="lines", line_color='#0E4001', name='average'), secondary_y=False)
#fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c_no_tectonic'],  mode="markers", marker_color="lightgreen",  name='d13C without tectonic', text=df_sorted['Name']),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_notec_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)



fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)

#fig.add_trace(go.Scatter(x=coefs_centered['age_absolute'], y=coefs_centered['d13c_slope'], mode="lines", line_color='pink', name='derivative'), secondary_y=True)
fig.add_trace(go.Scatter(x=tide['Date'], y=tide['Sea Level'], mode="lines", line_color='#F54927', name='Tide Gauge', line_width=1), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=False)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=True)) # Grid for secondary Y-axis)


for i, dates in enumerate(ENSO_match):
    fig.add_vrect(x0=dates[0], 
                  x1=dates[1], 
                  fillcolor=phase_color['C'], opacity=0.1)


#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

#for i, index in enumerate(blocks.index):
#    block = blocks.loc[index]
#    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
#                  x1=datetime_to_decimal_year(block['end']), 
#                  fillcolor=phase_color[block['value']], opacity=0.1)

#fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.975], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
#                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
#                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
#                        name='M8.1 EQ 2007 Apr'))

#fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

#fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d13C and tide gauge", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="Tide Gauge [mm]", secondary_y=True)

fig.show()
#fig.write_html('d13c_tidegauge.html')


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
#fig = make_subplots(specs=[[{"secondary_y": True}]])
fig = make_subplots()

fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d18o'],  mode="markers", marker_color="lightblue",  text=df_sorted['Name'],  name='d18O'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d18o_ma'], mode="lines", line_color='navy', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d18o_ma2'], mode="lines", line_color='orange', name='average'), secondary_y=False)



#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)

#fig.add_trace(go.Scatter(x=coefs_centered['age_absolute'], y=coefs_centered['d13c_slope'], mode="lines", line_color='pink', name='derivative'), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y=tide['Sea Level'], mode="lines", line_color='red', name='Tide Gauge'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=False)



#fig.update_layout(
#    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=True)) # Grid for secondary Y-axis)


for i, row in enumerate(dark_bands.itertuples()):
    fig.add_vrect(x0=row.age_absolute_start, 
                  x1=row.age_absolute_end, 
                  fillcolor=phase_color['C'], opacity=0.1)


#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

#for i, index in enumerate(blocks.index):
#    block = blocks.loc[index]
#    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
#                  x1=datetime_to_decimal_year(block['end']), 
#                  fillcolor=phase_color[block['value']], opacity=0.1)

fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[25.85], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
#                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
#                        name='M8.1 EQ 2007 Apr'))

#fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

#fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1800, height=700, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta O ^{18}$', secondary_y=False)
#fig.update_yaxes(title_text="Tide Gauge [mm]", secondary_y=True)

fig.show()
#fig.write_html('d18O.html')


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
#fig = make_subplots(specs=[[{"secondary_y": True}]])
fig = make_subplots()

fig.add_trace(go.Scatter(x=df_sorted.index, y=df_sorted['d18o'],  mode="markers", marker_color="lightblue",  text=df_sorted['Name'],  name='d18O'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted.index, y=df_sorted['d18o_ma'], mode="lines", line_color='navy', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted.index, y=df_sorted['d18o_ma2'], mode="lines", line_color='orange', name='average'), secondary_y=False)



#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)

#fig.add_trace(go.Scatter(x=coefs_centered['age_absolute'], y=coefs_centered['d13c_slope'], mode="lines", line_color='pink', name='derivative'), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y=tide['Sea Level'], mode="lines", line_color='red', name='Tide Gauge'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=False)



#fig.update_layout(
#    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=True)) # Grid for secondary Y-axis)


for i, row in enumerate(dark_bands.itertuples()):
    fig.add_vrect(x0=row.index_start, 
                  x1=row.index_end, 
                  fillcolor=phase_color['C'], opacity=0.1)


#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

#for i, index in enumerate(blocks.index):
#    block = blocks.loc[index]
#    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
#                  x1=datetime_to_decimal_year(block['end']), 
#                  fillcolor=phase_color[block['value']], opacity=0.1)

#fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[25.85], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
#                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
#                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
#                        name='M8.1 EQ 2007 Apr'))

#fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

#fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1800, height=700, title_text="d18O and dark bands", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta O ^{18}$', secondary_y=False)
#fig.update_yaxes(title_text="Tide Gauge [mm]", secondary_y=True)

fig.show()
#fig.write_html('d18O_dark-band_index.html')


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
#fig = make_subplots(specs=[[{"secondary_y": True}]])
fig = make_subplots()

fig.add_trace(go.Scatter(x=df_sorted.index, y=df_sorted['d13c'],  mode="markers", marker_color="lightgreen",  text=df_sorted['Name'],  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted.index, y=df_sorted['d13c_ma'], mode="lines", line_color='darkgreen', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted.index, y=df_sorted['d13c_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)



#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)

#fig.add_trace(go.Scatter(x=coefs_centered['age_absolute'], y=coefs_centered['d13c_slope'], mode="lines", line_color='pink', name='derivative'), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y=tide['Sea Level'], mode="lines", line_color='red', name='Tide Gauge'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=False)



#fig.update_layout(
#    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=True)) # Grid for secondary Y-axis)


for i, row in enumerate(dark_bands.itertuples()):
    fig.add_vrect(x0=row.index_start, 
                  x1=row.index_end, 
                  fillcolor=phase_color['C'], opacity=0.1)


#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

#for i, index in enumerate(blocks.index):
#    block = blocks.loc[index]
#    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
#                  x1=datetime_to_decimal_year(block['end']), 
#                  fillcolor=phase_color[block['value']], opacity=0.1)

#fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[25.85], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
#                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
#                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
#                        name='M8.1 EQ 2007 Apr'))

#fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

#fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1800, height=700, title_text="d13C and dark bands", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta C ^{13}$', secondary_y=False)
#fig.update_yaxes(title_text="Tide Gauge [mm]", secondary_y=True)

fig.show()
fig.write_html('d13C_dark-band_index.html')


In [ ]:
row

In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o'],  mode="markers", marker_color="lightblue",  name='d18O'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o_ma'], mode="lines", line_color='navy', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o_ma2'], mode="lines", line_color='orange', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_pdb'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o_std'], mode="lines", line_color='darkred', name='std'), secondary_y=True)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_std'], mode="lines", line_color='darkred', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)


#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

for i, index in enumerate(blocks.index):
    block = blocks.loc[index]
    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
                  x1=datetime_to_decimal_year(block['end']), 
                  fillcolor=phase_color[block['value']], opacity=0.1)

fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[25.85], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='N Gizo EQ 1936 Apr'))

fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[25.66], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE


fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[25.6], mode='markers', 
                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
                        name='M8.1 EQ 2007 Apr'))

#fig.add_annotation(x=1992.5, y=25.7, text="?", showarrow=False, font=dict(size=35, color="red"))

#fig.add_annotation(x=1991.4, y=25.6, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=20, ay=-50)

#fig.add_annotation(x=1974.3, y=25.6, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-20, ay=40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d18o", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta O ^{18}$', secondary_y=False)
fig.update_yaxes(title_text="Standard Deviation", secondary_y=True)

fig.show()
#fig.write_html('d18o_std.html')


In [ ]:
blue = "#4287f5"
orange = "#f58442"
green = "#42f5ad"
red = "#f54254"
yellow = "#e4f000"
purple = "#f000ec"
green2 = "#00f070"
blue2 = "#00c0f0"
pink = "#e37b7b"
colors = [blue, orange, green, red, yellow, purple, green2, blue2, pink]



fig = go.Figure()
for i, path in enumerate(paths):
    df_iter = df_sorted[df_sorted['path_name']==path]
    fig.add_trace( go.Scatter(x=df_iter['age_absolute'], y=df_iter['d18o'], mode='markers', marker=dict(color=colors[i]), name=path, text=df_iter['Name']))
    path_ma = df_iter['d18o'].rolling(window_size, center=True).mean()
    fig.add_trace(go.Scatter(x=df_iter['age_absolute'], y=path_ma, mode='lines',marker=dict(color=colors[i]), showlegend=False))
    
fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[26.235], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='N Gizo EQ 1936 Apr'))
fig.add_shape(type='line',
    x0=1936+(31+29+22)/365, x1=1936+(31+29+22)/365,
    y0=0, y1=1,  # Use yref='paper' for full height
    xref='x', yref='paper',
    line=dict(color="red", width=2, dash='dash'),
    name="EQ line"
)


for i, row in enumerate(dark_bands.itertuples()):
    fig.add_vrect(x0=row.age_absolute_start, 
                  x1=row.age_absolute_end, 
                  fillcolor=phase_color['C'], opacity=0.1)

    
fig.update_layout(width=1200, height=600, title_text="d18o", template='simple_white')
fig.show()
#fig.write_html('d18O_paths.html')

In [ ]:
blue = "#4287f5"
orange = "#f58442"
green = "#42f5ad"
red = "#f54254"
yellow = "#e4f000"
purple = "#f000ec"
green2 = "#00f070"
blue2 = "#00c0f0"
pink = "#e37b7b"
colors = [blue, orange, green, red, yellow, purple, green2, blue2, pink]



fig = go.Figure()
for i, path in enumerate(paths):
    df_iter = df_sorted[df_sorted['path_name']==path]
    fig.add_trace( go.Scatter(x=df_iter.index, y=df_iter['d18o'], mode='markers', marker=dict(color=colors[i]), name=path, text=df_iter['Name']))
    path_ma = df_iter['d18o'].rolling(window_size, center=True).mean()
    fig.add_trace(go.Scatter(x=df_iter.index, y=path_ma, mode='lines',marker=dict(color=colors[i]), showlegend=False))
    
#fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[26.235], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
#                        name='N Gizo EQ 1936 Apr'))
#fig.add_shape(type='line',
#    x0=1936+(31+29+22)/365, x1=1936+(31+29+22)/365,
#    y0=0, y1=1,  # Use yref='paper' for full height
#    xref='x', yref='paper',
#    line=dict(color="red", width=2, dash='dash'),
#    name="EQ line"
#)


for i, row in enumerate(dark_bands.itertuples()):
    fig.add_vrect(x0=row.index_start, 
                  x1=row.index_end, 
                  fillcolor=phase_color['C'], opacity=0.1)

    
fig.update_layout(width=1200, height=600, title_text="d18o", template='simple_white')
fig.show()
#fig.write_html('d18O_paths.html')

In [ ]:
blue = "#4287f5"
orange = "#f58442"
green = "#42f5ad"
red = "#f54254"
yellow = "#e4f000"
purple = "#f000ec"
green2 = "#00f070"
blue2 = "#00c0f0"
pink = "#e37b7b"
colors = [blue, orange, green, red, yellow, purple, green2, blue2, pink]



fig = go.Figure()
for i, path in enumerate(paths):
    df_iter = df_sorted[df_sorted['path_name']==path]
    fig.add_trace( go.Scatter(x=df_iter['age_absolute'], y=df_iter['d13c'], mode='markers', marker=dict(color=colors[i]), name=path, text=df_iter['Name']))
    path_ma = df_iter['d13c'].rolling(window_size, center=True).mean()
    fig.add_trace(go.Scatter(x=df_iter['age_absolute'], y=path_ma, mode='lines',marker=dict(color=colors[i]), showlegend=False))
    
fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.41], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='N Gizo EQ 1936 Apr'))
fig.add_shape(type='line',
    x0=1936+(31+29+22)/365, x1=1936+(31+29+22)/365,
    y0=0, y1=1,  # Use yref='paper' for full height
    xref='x', yref='paper',
    line=dict(color="red", width=2, dash='dash'),
    name="EQ line"
)
fig.update_layout(width=1200, height=600, title_text="d13c", template='simple_white')
fig.show()
#fig.write_html('d18O_paths.html')

In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d18o'],  mode="markers", marker_color="lightblue", text=df_sorted['Name'], name='d18O'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o_ma'], mode="lines", line_color='navy', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o_ma2'], mode="lines", line_color='orange', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_pdb'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o_std'], mode="lines", line_color='darkred', name='std'), secondary_y=True)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_std'], mode="lines", line_color='darkred', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)


#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

for i, index in enumerate(blocks.index):
    block = blocks.loc[index]
    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
                  x1=datetime_to_decimal_year(block['end']), 
                  fillcolor=phase_color[block['value']], opacity=0.1)

fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[25.85], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='N Gizo EQ 1936 Apr'))

fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[25.66], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE


fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[25.6], mode='markers', 
                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
                        name='M8.1 EQ 2007 Apr'))

#fig.add_annotation(x=1992.5, y=25.7, text="?", showarrow=False, font=dict(size=35, color="red"))

fig.add_annotation(x=1991.4, y=25.6, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=20, ay=-50)

fig.add_annotation(x=1974.3, y=25.6, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-20, ay=40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d18o", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta O ^{18}$', secondary_y=False)
fig.update_yaxes(title_text="Standard Deviation", secondary_y=True)

fig.show()
#fig.write_html('d18o_std.html')


In [ ]:
len(df_sorted) + len(fred)

In [ ]:
fig, axes = plt.subplots(3, figsize=(10,7), sharex=True,height_ratios=[3,1,1], gridspec_kw={'hspace':0})

ax = axes[0]

ax.plot(df_resampled['age_absolute'], df_resampled['d18o'], ".", color='lightblue', alpha=0.7, label=r'$\delta^{18}O$' )
ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma'],  color='navy', lw=0.75 )
ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['decimal_year'], fred['d18o_pdb'], ".", color='#40E0D0', alpha=0.3,  )
ax.plot(fred['decimal_year'], fred['d18o_ma'],  color='#1AC4A8', lw=0.75 )


ax.vlines([1936+(31+29+22)/365, 2007+(31+29+31)/365],
          ymin=20, ymax=30, color=['#F54927','#2D8708'],label='1936 EQ')

ax.set_ylim(25, 26.57)

ax.grid()
ax.legend(loc='lower left')



ax = axes[1]

ax.plot(df_resampled['age_absolute'], df_resampled['d18o_std'], color='darkred' , label='std d18O'  )
ax.plot(fred['decimal_year'], fred['d18o_std'], '--', color='darkred' , label='fred std d18O'  )

#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=-1, ymax=1, color=['#F54927','#F5C827','#2D8708'])

ax.vlines([1936+(31+29+22)/365,  2007+(31+29+31)/365], ymin=-1, ymax=1, color=['#F54927','#2D8708'])

ax.set_ylim(-0.05,0.35)
ax.grid()
ax.legend(loc='lower left')



ax = axes[2]

ax.plot(coefs_centered['age_absolute'], coefs_centered['d13c_slope'], color='pink' , label='moving OLS'  )

ax.vlines([1936+(31+29+22)/365,  2007+(31+29+31)/365], ymin=-1, ymax=1, color=['#F54927','#2D8708'])


ax.set_ylim(-0.5,0.5)
ax.grid()
ax.legend(loc='lower left')

#fig.savefig('d18O_std_OLS.pdf')

In [ ]:
primary_font = 15
secondary_font = 12

fig, ax = plt.subplots(1, figsize=(12,6))

ax.plot(df_resampled['age_absolute'], df_resampled['d13c'], ".", color='lightgreen', alpha=0.7, label=r'$\delta^{13}$C' )
ax.plot(df_resampled['age_absolute'], df_resampled['d13c_ma'],  color='green', lw=1 )
ax.plot(df_resampled['age_absolute'], df_resampled['d13c_ma2'],  color='darkgreen', lw=2.5 )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['decimal_year'], fred['d13c'], ".", color='#40E0D0', alpha=0.7, label='Earlier Samples' )
ax.plot(fred['decimal_year'], fred['d13c_ma'],  color='blue', lw=1.2 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

plt.axvline(1936+(31+29+22)/365, color='#A87902',linestyle='--', label='M7.1 EQ 1936 Apr')
plt.axvline(2007+(31+29+31)/365, color='#F50727', label='M8.1 EQ 2007 Apr')

ax.set_ylabel(r'$\delta^{13}C$ [PDB]', fontdict={'fontsize': secondary_font})
ax.set_xlabel('Year', fontdict={'fontsize': secondary_font})

ax.set_ylim(-3, 0.5)
ax.set_xlim(1927, 2012)

ax.grid()
ax.legend()
#ax.legend(bbox_to_anchor=(1, 1))



fig.savefig('d13C_overlook.pdf')
fig.savefig('d13C_overlook.png')

In [ ]:
primary_font = 15
secondary_font = 12

fig, ax = plt.subplots(1, figsize=(12, 5))

ax.plot(df_resampled['age_absolute'], df_resampled['d18o'], ".", color='lightblue', alpha=0.7, label=r'$\delta^{18}$O' )
ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma'],  color='navy', lw=0.75 )
ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['decimal_year'], fred['d18o_pdb'], ".", color='#40E0D0', alpha=0.7, label='Earlier Samples' )
ax.plot(fred['decimal_year'], fred['d18o_ma'],  color='#1AC4A8', lw=0.75 )


plt.axvline(1936+(31+29+22)/365, color='#A87902',linestyle='--', label='M7.1 EQ 1936 Apr')
plt.axvline(2007+(31+29+31)/365, color='#F50727', label='M8.1 EQ 2007 Apr')


ax.set_ylabel(r'$\delta^{18}O$ [SMOW]', fontdict={'fontsize': secondary_font})
ax.set_xlabel('Year', fontdict={'fontsize': secondary_font})

ax.set_ylim(25, 26.2)

ax.grid()
ax.legend(loc='lower left')

#fig.savefig('d18o_overlook.png')
#fig.savefig('d18o_overlook.pdf')

In [ ]:
primary_font = 17
secondary_font = 15

fig, axes = plt.subplots(2, figsize=(12,10), sharex=True, height_ratios=(5,4), gridspec_kw={'hspace':0})

ax = axes[0]
ax.plot(df_resampled['age_absolute'], df_resampled['d13c'], ".", color='lightgreen', alpha=0.7, label=r'$\delta^{13}$C' )
ax.plot(df_resampled['age_absolute'], df_resampled['d13c_ma'],  color='green', lw=1 )
ax.plot(df_resampled['age_absolute'], df_resampled['d13c_ma2'],  color='red', lw=2.5 )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['decimal_year'], fred['d13c'], ".", color='#40E0D0', alpha=0.7, label='Prior Results' )
ax.plot(fred['decimal_year'], fred['d13c_ma'],  color='blue', lw=1.2 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

ax.axvline(1936+(31+29+22)/365, color='#A87902',linestyle='--', label='M7.1 EQ 1936 Apr')
ax.axvline(2007+(31+29+31)/365, color='#F50727', label='M8.1 EQ 2007 Apr')

ax.set_ylabel(r'$\delta^{13}C$ [PDB]', fontdict={'fontsize': secondary_font})
#ax.set_xlabel('Year', fontdict={'fontsize': secondary_font})

ax.set_ylim(-3, 0.5)
ax.set_xlim(1927, 2012)

ax.tick_params(axis='y', labelsize=secondary_font*0.85)

ax.grid()
ax.legend()
#ax.legend(bbox_to_anchor=(1, 1))

ax = axes[1]
ax.plot(df_resampled['age_absolute'], df_resampled['d18o'], ".", color='lightblue', alpha=0.7, label=r'$\delta^{18}$O' )
ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma'],  color='navy', lw=0.75 )
ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['decimal_year'], fred['d18o_pdb'], ".", color='#40E0D0', alpha=0.7, label='Prior Results' )
ax.plot(fred['decimal_year'], fred['d18o_ma'],  color='#1AC4A8', lw=0.75 )


plt.axvline(1936+(31+29+22)/365, color='#A87902',linestyle='--', label='M7.1 EQ 1936 Apr')
plt.axvline(2007+(31+29+31)/365, color='#F50727', label='M8.1 EQ 2007 Apr')


ax.set_ylabel(r'$\delta^{18}O$ [SMOW]', fontdict={'fontsize': secondary_font})
ax.set_xlabel('Year', fontdict={'fontsize': secondary_font})

ax.set_ylim(25, 26.2)


yticks = ax.get_yticks()
# Get current labels
yticklabels = ax.get_yticklabels()

# Create a new list of labels, setting the top one (last in list) to a blank string
new_labels = [label.get_text() for label in yticklabels]
if new_labels:
    new_labels[-1] = "" # Set the last label to an empty string

# Apply the new labels
ax.set_yticklabels(new_labels)
ax.tick_params(axis='both', labelsize=secondary_font*0.85)


ax.grid()
ax.legend(loc='lower left')


fig.savefig('d13C_d18O_overlook.pdf')
fig.savefig('d13C_d18O_overlook.png')

In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c'],  mode="markers", marker_color="lightgreen",  name='d13C', text=df_sorted['Name']),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)



fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)

#fig.add_trace(go.Scatter(x=coefs_centered['age_absolute'], y=coefs_centered['d13c_slope'], mode="lines", line_color='pink', name='derivative'), secondary_y=True)


fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=True)) # Grid for secondary Y-axis)

#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

#for i, index in enumerate(blocks.index):
#    block = blocks.loc[index]
#    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
#                  x1=datetime_to_decimal_year(block['end']), 
#                  fillcolor=phase_color[block['value']], opacity=0.1)

fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.975], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
                        name='M8.1 EQ 2007 Apr'))

#fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

#fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1000, height=400, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="Standard Deviation", secondary_y=True)

fig.show()
#fig.write_html('d13c_std.html')


In [ ]:
import matplotlib.ticker as ticker


In [ ]:
fig, axes = plt.subplots(3, figsize=(4,6), sharex=True,gridspec_kw={'hspace':0} )

for i in range(3):
    ax = axes[i]
    if i==0:
        ax.plot(1/d18o_fft_freq[1:], fft_noramlize(d18o_fft[1:])**2,label=r'$\delta^{18}$O')
    if i==1:
        ax.plot(1/d13c_fft_freq[1:], fft_noramlize(d13c_fft[1:])**2, color='green',label=r'$\delta^{13}$C')
    if i==2:
        ax.plot(1/enso_fft_freq[1:], fft_noramlize(enso_fft[1:])**3, color='gray', label='ENSO Index')
    ax.legend(loc='center left')  
    #days_freq = 271
    #ax.vlines(days_freq, ymin=-100, ymax=2500, color='purple', linestyles='dotted', alpha=0.4, label='{:} days'.format(days_freq))
    #days_freq = 281
    #ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
    ax.vlines(365, ymin=-100, ymax=2500, color='red', linestyles='solid', alpha=0.4, label='365 days')



    ax.set_xlim(150,500)
#ax.set_xlim(200,3600)
#ax.set_xscale('log')
    ax.set_ylim(0,0.001)
    plt.yticks([0,0.001, 0.002])
axes[0].set_ylim(0,0.000001)
#plt.yticks([0.5e-6, 1e-6])

ax = axes[1]
ax.yaxis.tick_right()
#ax.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:.2e}"))
ax.set_ylabel('Normalized FFT'+'\n'+'Amplitude', labelpad=25)
#ax.yaxis.set_label_position("right") # Also moves the label position


axes[0].set_title('Frequency Spectrum')
axes[1].legend(loc='center left')
axes[2].set_xlabel('Period [days]')

#ax.set_title('Moving Standard Deviation FFT')
plt.show()



fig.savefig('fft_d13c_d18o.pdf', bbox_inches='tight', pad_inches=0.15)
fig.savefig('fft_d13c_d18o.png', bbox_inches='tight', pad_inches=0.15)



In [ ]:
fig, axes = plt.subplots(3, figsize=(6,4), sharex=True,gridspec_kw={'hspace':0} )

for i in range(3):
    ax = axes[i]
    if i==0:
        ax.plot(1/d18o_std_fft_freq[1:], fft_noramlize(d18o_std_fft[1:]),label=r'$\delta^{18}O$')
    if i==1:
        ax.plot(1/d13c_std_fft_freq[1:], fft_noramlize(d13c_std_fft[1:]), color='green',label=r'$\delta^{13}C$')
    if i==2:
        ax.plot(1/enso_fft_freq[1:], fft_noramlize(enso_fft[1:]), color='gray', label='enso')
    
    ax.legend(loc='upper left')  
    days_freq = 271
    ax.vlines(days_freq, ymin=-100, ymax=2500, color='purple', linestyles='dotted', alpha=0.4, label='{:} days'.format(days_freq))
    days_freq = 281
    ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
    ax.vlines(365, ymin=-100, ymax=2500, color='red', linestyles='solid', alpha=0.4, label='1 Y')

    days_freq = 746    
    ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
    
    
    ax.set_xlim(100,300)
    #ax.set_xlim(200,3600)
    #ax.set_xscale('log')
    #plt.yticks([0,0.001])
    ax.set_ylim(0,0.005)

axes[2].set_ylim(0,0.072)
ax.set_xlabel('Period [days]')
axes[1].set_ylabel('Normalized FFT Amplitude')
axes[0].set_title('Frequency Spectrum ')
axes[1].legend(loc='upper left')  
#ax.legend(loc=[500,0])
#ax.set_title('Moving Standard Deviation FFT')
plt.show()
#fig.savefig('std_fft.pdf')


In [ ]:
# Ensure df_resampled is sorted by date
df = df_resampled[['d13c','d18o', 'd13c_no_tectonic', 'd18o_no_tectonic', 'd13c_notec_ma']].copy()
df = df.sort_index()

# Target dates to align with
target_dates = pd.to_datetime(enso.loc[:685, 'Date'])

# Interpolate to get values at those dates
aligned = df.reindex(df.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned['age_absolute'] = np.array([datetime_to_decimal_year(x) for x in pd.to_datetime(aligned.index)])
aligned['ENSO'] = enso.loc[:685, 'ENSO'].values
aligned_tidegauge = tide.reindex(tide.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned = aligned.join(aligned_tidegauge['Sea Level'].dropna())
aligned.loc['2005-01-01':'2008-01-01']

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c'],  mode="markers", marker_color="lightgreen",  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=aligned['age_absolute'], y=aligned['d13c'], mode="markers", line_color='red', name='enso_aligned'), secondary_y=False)

fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="markers", line_color='gray', name='ENSO'), secondary_y=True)


In [ ]:
correlation_matrix = aligned.loc['1969-01-01':'1975-01-01'].corr()

print(correlation_matrix)

In [ ]:
correlation_matrix = aligned.corr()

# Create a heatmap
fig = plt.figure(figsize=(6, 4))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt=".2f")
plt.title("Correlation Matrix Heatmap")
plt.tight_layout()
plt.show()
#fig.savefig('corr_matrix.png')

In [ ]:
#---------------------------------------FFT-----------------------------------------------------
d13c_aligned_fft_freq, d13c_aligned_fft = FFT_res(aligned['d13c'], 30)
d18o_aligned_fft_freq, d18o_aligned_fft = FFT_res(aligned['d18o'], 30)
#-----------------------------------------------------------------------------------------------

In [ ]:
fig, axes = plt.subplots(3, figsize=(6,4), sharex=True , gridspec_kw={'hspace':0} )

for i in range(3):
    ax = axes[i]
    if i==0:
        ax.plot(1/d18o_aligned_fft_freq[1:], fft_noramlize(d18o_aligned_fft[1:]),label=r'$\delta^{18}O$')
    if i==1:
        ax.plot(1/d13c_aligned_fft_freq[1:], fft_noramlize(d13c_aligned_fft[1:]), color='green',label=r'$\delta^{13}C$')
    if i==2:
        ax.plot(1/enso_fft_freq[1:], fft_noramlize(enso_fft[1:]), color='gray', label='enso')
    
    ax.legend(loc='upper right')  
    days_freq = 271
    #ax.vlines(days_freq, ymin=-100, ymax=2500, color='purple', linestyles='dotted', alpha=0.4, label='{:} days'.format(days_freq))
    days_freq = 281
    #ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
    days_freq = 746    
    #ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
    
    ax.vlines(365, ymin=-100, ymax=2500, color='red', linestyles='solid', alpha=0.4, label='1 Y')
    
    ax.set_xlim(300,700)
    #ax.set_xlim(200,3600)
    #ax.set_xscale('log')
    #plt.yticks([0,0.001])
    
    
axes[0].set_ylim(0,0.001)
axes[1].set_ylim(0,0.04)
axes[2].set_ylim(0,0.4)
ax.set_xlabel('Period [days]')
axes[1].set_ylabel('Normalized FFT Amplitude')
axes[0].set_title('Frequency Spectrum ')
axes[1].legend('upper right')
#ax.set_title('Moving Standard Deviation FFT')
plt.show()
#fig.savefig('std_fft.pdf')


In [ ]:
fig, ax = plt.subplots(1, figsize=(10,5))

ax.plot(df_resampled['age_absolute'], df_resampled['d18o'], ".", color='lightblue', alpha=0.7, label=r'$\delta^{18}O$' )
ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma'],  color='navy', lw=0.75 )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['decimal_year'], fred['d18o_pdb'], ".", color='#40E0D0', alpha=0.3,  )
ax.plot(fred['decimal_year'], fred['d18o_ma'],  color='#1AC4A8', lw=0.75 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])



plt.axvline(1936+(31+29+31+19)/365, color='#F54927', label='M7.1 EQ 1936 Apr')

x, y, p= line_fit(df_resampled.loc[:'1936-07-30','age_absolute'].values, 
               df_resampled.loc[:'1936-07-30','d18o'].values)
ax.plot(x, y, '--', color='#FF9724', linewidth=2.5, label=f'n = {p[0]:.3f}'+r' $\delta^{18}O$ per year')


x, y, p= line_fit(df_resampled.loc['1936-07-30':'1944-12','age_absolute'].values, 
               df_resampled.loc['1936-07-30':'1944-12','d18o'].values)
ax.plot(x, y, '--', color='#90E84D', linewidth=2.5, label=f'n = {p[0]:.3f}'+r' $\delta^{18}O$ per year')


ax.set_ylabel(r'$\delta^{18}O$ [SMOW]', fontdict={'fontsize': 15})
ax.set_xlabel('Year', fontdict={'fontsize': 15})

ax.set_ylim(25, 26.57)
ax.set_xlim(1927, 1945)

ax.set_xticks(np.arange(1927,1946))
plt.xticks(rotation=45)
ax.grid()
ax.legend(loc='lower left')



fig.savefig('d18O_1936_closeup_trends.pdf', bbox_inches='tight', pad_inches=0)
fig.savefig('d18O_1936_closeup_trends.png', bbox_inches='tight', pad_inches=0)

In [ ]:
fig, ax = plt.subplots(1, figsize=(12,6))

ax.plot(df_resampled['age_absolute'], df_resampled['d13c'], ".", color='lightgreen', alpha=0.7, label=r'$\delta^{13}C$' )
ax.plot(df_resampled['age_absolute'], df_resampled['d13c_ma2'],  color='darkgreen', lw=2.5 )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['decimal_year'], fred['d13c'], ".", color='#40E0D0', alpha=0.7, label='Eearlier Samples'  )
ax.plot(fred['decimal_year'], fred['d13c_ma'],  color='#1AC4A8', lw=0.75 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

plt.axvline(1936+(31+29+22)/365, color='#A87902',linestyle='--', label='M7.1 EQ 1936 Apr')
plt.axvline(2007+(31+29+31)/365, color='#F50727', label='M8.1 EQ 2007 Apr')



#date1 = '1936-04-19'
date1 = '1936-08-30'
#date2 = '1944-06'
date2 = '1941-06'
date2a = '1949-06'
#date3 = '1966-02'
date3 = '1963-01'
date4 = '1976-06'
date5 = '1987-06'
date6 = '2007-01'


                   ###########  1
x, y, p= line_fit(df_resampled.loc[:date1,'age_absolute'].values, 
               df_resampled.loc[:date1,'d13c'].values)
ax.plot(x, y, '-', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{18}O$ ‰ per year')

                   ###########  2
x, y, p= line_fit(df_resampled.loc[date1:date2,'age_absolute'].values, 
               df_resampled.loc[date1:date2,'d13c'].values)
ax.plot(x, y, '-', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{18}O$ ‰ per year')

x, y, p= line_fit(df_resampled.loc[date2:date2a,'age_absolute'].values, 
               df_resampled.loc[date2:date2a,'d13c'].values)
ax.plot(x, y, '-', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{18}O$ ‰ per year')

                   ###########  3
x, y, p= line_fit(df_resampled.loc[date2a:date3,'age_absolute'].values, 
               df_resampled.loc[date2a:date3,'d13c'].values)
ax.plot(x, y, '-', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{18}O$ ‰ per year')

                   ###########  4
x, y, p= line_fit(df_resampled.loc[date3:date4,'age_absolute'].values, 
               df_resampled.loc[date3:date4,'d13c'].values)
ax.plot(x, y, '--', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{18}O$ ‰ per year')

                   ###########  5
x, y, p= line_fit(df_resampled.loc[date4:date5,'age_absolute'].values, 
               df_resampled.loc[date4:date5,'d13c'].values)
ax.plot(x, y, '--', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{18}O$ ‰ per year')

                   ###########  6
x, y, p= line_fit(df_resampled.loc[date5:date6,'age_absolute'].values, 
               df_resampled.loc[date5:date6,'d13c'].values)
ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{18}O$ ‰ per year')

ax.set_ylabel(r'$\delta^{13}C$ [PDB]', fontdict={'fontsize': 15})
ax.set_xlabel('Year', fontdict={'fontsize': 15})

ax.set_ylim(-3, 0.5)
ax.set_xlim(1927, 2012)

ax.grid()
#ax.legend(loc='lower left')
#ax.legend(bbox_to_anchor=(1, 1))
ax.legend(
    loc='upper right',
    #bbox_to_anchor=(0.5, -0.15),  # Centered below the plot
    ncol=2,                       # 3 columns
    frameon=False,               # Optional: remove box
    columnspacing=1.5,
    handletextpad=0.4,
    fontsize=secondary_font*0.75
)

ax.tick_params(axis='both', labelsize=secondary_font*0.85)

fig.savefig('d13C_trens.pdf')
fig.savefig('d13C_trens.png')

In [ ]:
def d13c_to_RSL(d13c, n, m):
    res = d13c*n+m
    return res

In [ ]:
df_resampled['RSL_d13c'] = d13c_to_RSL(df_resampled['d13c'], 35.546, -65.672)
df_resampled['RSL_d13c_ma'] = d13c_to_RSL(df_resampled['d13c_ma'], 35.546, -65.672)
df_resampled['RSL_d13c_ma2'] = d13c_to_RSL(df_resampled['d13c_ma2'], 35.546, -65.672)
fred_resampled['RSL_fred'] = d13c_to_RSL(fred_resampled['d13c'], 35.546, -65.672)


In [ ]:
fred_resampled['d13c']

In [ ]:
fig, ax = plt.subplots(1, figsize=(12,6))

ax.plot(df_resampled['age_absolute'], df_resampled['RSL_d13c'], ".", color='lightgreen', alpha=0.7, label=r'$\delta^{13}C$' )
ax.plot(df_resampled['age_absolute'], df_resampled['RSL_d13c_ma2'],  color='darkgreen', lw=2.5 )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )

ax.plot(fred_resampled['decimal_year'], fred_resampled['RSL_fred'], ".", color='#40E0D0', alpha=0.7, label='Prior Results' )
#ax.plot(fred_resampled['decimal_year'], fred_resampled['RSL_fred_ma'],  color='#1AC4A8', lw=0.75 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

plt.axvline(1936+(31+29+22)/365, color='#A87902',linestyle='--', label='M7.1 EQ 1936 Apr')
plt.axvline(2007+(31+29+31)/365, color='#F50727', label='M8.1 EQ 2007 Apr')



#date1 = '1936-04-19'
date1 = '1936-08-30'
#date2 = '1944-06'
date2 = '1941-06'
date2a = '1949-06'
#date3 = '1966-02'
date3 = '1963-01'
date4 = '1976-06'
date5 = '1987-06'
date6 = '2007-01'


                   ###########  1
x, y, p= line_fit(df_resampled.loc[:date1,'age_absolute'].values, 
               df_resampled.loc[:date1,'RSL_d13c'].values)
ax.plot(x, y, '-', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+'cm/a')

                   ###########  2
x, y, p= line_fit(df_resampled.loc[date1:date2,'age_absolute'].values, 
               df_resampled.loc[date1:date2,'RSL_d13c'].values)
ax.plot(x, y, '-', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+'cm/a')

x, y, p= line_fit(df_resampled.loc[date2:date2a,'age_absolute'].values, 
               df_resampled.loc[date2:date2a,'RSL_d13c'].values)
ax.plot(x, y, '-', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+'cm/a')

                   ###########  3
x, y, p= line_fit(df_resampled.loc[date2a:date3,'age_absolute'].values, 
               df_resampled.loc[date2a:date3,'RSL_d13c'].values)
ax.plot(x, y, '-', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+'cm/a')
                   ###########  4
x, y, p= line_fit(df_resampled.loc[date3:date4,'age_absolute'].values, 
               df_resampled.loc[date3:date4,'RSL_d13c'].values)
ax.plot(x, y, '--', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+'cm/a')

                   ###########  5
x, y, p= line_fit(df_resampled.loc[date4:date5,'age_absolute'].values, 
               df_resampled.loc[date4:date5,'RSL_d13c'].values)
ax.plot(x, y, '--', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+'cm/a')

                   ###########  6
x, y, p= line_fit(df_resampled.loc[date5:date6,'age_absolute'].values, 
               df_resampled.loc[date5:date6,'RSL_d13c'].values)
ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+'cm/a')

ax.set_ylabel(r'RSL based on $\delta^{13}C$ [cm]', fontdict={'fontsize': 15})
ax.set_xlabel('Year', fontdict={'fontsize': 15})

#ax.set_ylim(-3, 0.5)
ax.set_xlim(1927, 2012)

ax.grid()
#ax.legend(loc='lower left')
#ax.legend(bbox_to_anchor=(1, 1))
ax.legend(
    loc='upper right',
    #bbox_to_anchor=(0.5, -0.15),  # Centered below the plot
    ncol=2,                       # 3 columns
    frameon=False,               # Optional: remove box
    columnspacing=1.5,
    handletextpad=0.4,
    fontsize=secondary_font*0.85
)

ax.tick_params(axis='both', labelsize=secondary_font*0.85)

fig.savefig('d13C_trens.pdf')
fig.savefig('d13C_trens.png')

In [ ]:
df_iter =  df_resampled.loc[:date1]

fig, ax = plt.subplots(1 ,figsize=(8, 5))


ax.plot(df_iter['d18o'], df_iter['d13c'], '.')

x, y, p= line_fit(df_iter['d18o'].values, df_iter['d13c'].values)

ax.plot(x, y, '--', color='#90E84D', linewidth=2.5, label=f'n = {p[0]:.3f}')
ax.set_xlabel(r'$\delta^{18}O$')
ax.set_ylabel(r'$\delta^{13}C$')

ax.legend()